# Alpha scheduled codegen: a Jupyter walkthrough

Mirrors `docs/scheduled-codegen-design.md` §5.2's own worked example end to end: parse a
prefix-sum program, normalize it (hoisting its `reduce` into its own pair of statements),
inspect it, attach an explicit target mapping (schedule), and generate C.

This notebook is also a regression fixture, checked with `nbval`
(`pytest --nbval alpha-py/notebooks/prefix_scan.ipynb` from the repo root) — every cell's
saved output below is real, not illustrative; a code change that alters any of `alpha`'s
`__repr__` output or generated C will show up as a diff here.


In [1]:
import alpha

## Reading the program

`%%alpha <var>` parses the cell body as Alpha source and binds an `alpha.System` to `<var>`.

In [2]:
%%alpha sys
affine PrefixScan [N]->{:N>0}
    inputs  X: [N]
    outputs Y: [N]
    let Y[i] = reduce(+, [j], {:j<=i}: X[j]);
.

`System.__repr__` works immediately, no normalization needed: one entry per variable, at its
identity schedule, `reduce` still nested inside `Y`'s own equation (no statement split yet).

In [3]:
sys

[N] -> { Y[__rect0] -> [__rect0' = __rect0] : 0 <= __rect0 < N }

## Normalizing

`alpha.normalize` hoists the `reduce` into its own `Y_NR__init`/`Y_NR__reduce` statement pair
(§4.2) and returns a new `NormalizedSystem` — `sys` itself is untouched.

In [4]:
norm = alpha.normalize(sys)
norm

[N] -> { Y[__rect0] -> [__rect0' = __rect0, 0] : 0 <= __rect0 < N; Y_NR__init[i0] -> [i0, 0] : 0 <= i0 < N; Y_NR__reduce[i, j] -> [i' = i, j' = j] : i < N and 0 <= j <= i and j < N }

## Scheduling

`%%schedule <var> <source-system-var>` parses its cell body as a target mapping (§6), validates
it, checks legality against `norm`'s real dependences (§7), and binds a new `ScheduledSystem`.

In [5]:
%%schedule sched norm
{ Y_NR__init[i] -> [i, 0, 0]; Y_NR__reduce[i,j] -> [i, 1, j]; Y[i] -> [i, 2, 0]; }

In [6]:
sched

[N] -> { Y_NR__reduce[i, j] -> [i, 1, j] : i < N and 0 <= j <= i and j < N; Y_NR__init[i] -> [i, 0, 0] : 0 <= i < N; Y[i] -> [i, 2, 0] : 0 <= i < N }

## Generating C

`alpha.generate` runs `ScheduledC` and returns the generated C source as a `str`.

In [7]:
code = alpha.generate(sched)
print(code)

// This code was auto-generated by alphac (alpha-rs).
// ScheduledC backend — a user-supplied target mapping controls loop order.

#include <float.h>
#include <limits.h>
#include <math.h>
#include <stdbool.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

// Function Macros
#define ceild(n,d) ((int)ceil(((double)(n))/((double)(d))))
#define floord(n,d) ((int)floor(((double)(n))/((double)(d))))
#define div(a,b) (ceild((a),(b)))
#define max(a,b) (((a)>(b))?(a):(b))
#define min(a,b) (((a)<(b))?(a):(b))
#define mallocCheck(v,s) if ((v) == NULL) { printf("Failed to allocate memory for variable: %s\n", (s)); exit(-1); }

// Global Variables
static long N;
static float* X;
static float* Y;
static float* Y_NR;
static long Y_NR_min0;
static long Y_NR_size0;

// Memory Macros
#define X(i0) X[i0]
#define Y(i0) Y[i0]
#define Y_NR(i0) Y_NR[((i0) - (Y_NR_min0)) * (1)]

// Function Declarations
void PrefixScan(long _local_N, float* _local_X, float* _local_Y);

void PrefixScan(long _local

## The legality check, exercised

An *omitted* target mapping means every statement gets its own identity schedule (§6) — for a
program with a real `reduce` dependency like this one, that's illegal: nothing guarantees
`Y_NR__init`/`Y_NR__reduce` run in the right order relative to each other or to `Y`. `%%schedule`
(and `alpha.generate` on a bare `NormalizedSystem`) reject it with `alpha.ScheduleError` rather
than silently generating wrong code.

In [8]:
try:
    alpha.generate(norm)
except alpha.ScheduleError as e:
    print(f"ScheduleError: {e}")

ScheduleError: 'Y' reads 'Y_NR__reduce' but the schedule doesn't guarantee the producer instance runs strictly before the consumer instance that reads it (§7.2)
